In [3]:
# %%
import os
import glob
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# font
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Roboto Condensed', 'DejaVu Sans']
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 10

ALPHA = 0.05


def cetak_uji_hipotesis(
    judul,
    h0,
    h1,
    nama_stat,
    nilai_stat,
    kritis_str,
    p_val,
    alpha=ALPHA,
    ket_tolak="",
    ket_gagal_tolak=""
):
    tolak = p_val < alpha
    keputusan = "Tolak H0" if tolak else "Gagal Tolak H0"
    tanda = "<" if tolak else ">="
    kesimpulan = ket_tolak if tolak else ket_gagal_tolak

    print("-" * 88)
    print(f"Uji Hipotesis: {judul}")
    print(f"  H0: {h0}")
    print(f"  H1: {h1}")
    print(f"  Taraf signifikansi: alpha = {alpha}")
    print(f"  Statistik uji: {nama_stat} = {nilai_stat:.4f}")
    print(f"  Titik kritis: {kritis_str}")
    print(f"  p-value: {p_val:.4e}")
    print(f"  Keputusan: {keputusan} (karena p-value {tanda} {alpha})")
    print(f"  Kesimpulan: {kesimpulan}")

In [5]:
# dataset
path = kagglehub.dataset_download("joebeachcapital/top-500-hollywood-movies-of-all-time")
csv_files = glob.glob(os.path.join(path, "*.csv"))
df_raw = pd.read_csv(csv_files[0])
df_raw.columns = [c.strip() for c in df_raw.columns]

# kolom
df = pd.DataFrame({
    'title': df_raw['title'].astype(str).str.strip(),
    'pendapatan_m': pd.to_numeric(df_raw['worldwide gross (m)'], errors='coerce'),
    'budget_m': pd.to_numeric(df_raw['budget  (millions)'], errors='coerce'),
    'tahun': pd.to_numeric(df_raw['year'], errors='coerce'),
    'genre': df_raw['horror'].astype(str).str.strip().str.lower()
}).dropna().sort_values(by=['tahun', 'title']).reset_index(drop=True)

# variabel dummy genre: 0 = non horror (baseline), 1 = horror
df['D_horror'] = (df['genre'] == 'horror').astype(float)

# pengelompokan tahun per dekade, pembentukan k-1 = 5 variabel dummy
# baseline = '2010s'
df['dekade'] = ((df['tahun'] // 10) * 10).astype(int).astype(str) + 's'
urutan_dekade = ['2010s', '1970s', '1980s', '1990s', '2000s', '2020s']
df['dekade_cat'] = pd.Categorical(df['dekade'], categories=urutan_dekade, ordered=False)

dekade_dummies = pd.get_dummies(df['dekade_cat'], prefix='D_dekade', drop_first=True, dtype=float)
df = pd.concat([df, dekade_dummies], axis=1)

# vektor respon Y, matriks desain X
n_obs = len(df)
Y = df['pendapatan_m'].values
dummy_dekade_cols = list(dekade_dummies.columns)
X_cols = ['budget_m', 'D_horror'] + dummy_dekade_cols

feature_names = [
    'Intersep',
    'Budget (juta USD)',
    'Dummy Horror (1=Horror)',
    'Dummy Dekade 1970s',
    'Dummy Dekade 1980s',
    'Dummy Dekade 1990s',
    'Dummy Dekade 2000s',
    'Dummy Dekade 2020s'
]
X_mat = np.column_stack([np.ones(n_obs), df[X_cols].values])

print(f"Jumlah observasi (n): {n_obs}")
print("\nDistribusi Genre (0=not horror, 1=horror):")
print(df['D_horror'].value_counts().sort_index())
print("\nDistribusi Dekade (Grup dasar = 2010s):")
print(df['dekade'].value_counts().sort_index())
display(df[['title', 'pendapatan_m', 'budget_m', 'tahun', 'D_horror'] + dummy_dekade_cols].head())

Jumlah observasi (n): 562

Distribusi Genre (0=not horror, 1=horror):
D_horror
0.0    513
1.0     49
Name: count, dtype: int64

Distribusi Dekade (Grup dasar = 2010s):
dekade
1970s      4
1980s     15
1990s     59
2000s    142
2010s    312
2020s     30
Name: count, dtype: int64


,title,pendapatan_m,budget_m,tahun,D_horror,D_dekade_1970s,D_dekade_1980s,D_dekade_1990s,D_dekade_2000s,D_dekade_2020s
0,The Exorcist,441.0,12.0,1973,1.0,1.0,0.0,0.0,0.0,0.0
1,Jaws,477.0,9.0,1975,1.0,1.0,0.0,0.0,0.0,0.0
2,Star Wars,775.0,11.0,1977,0.0,1.0,0.0,0.0,0.0,0.0
3,Grease,396.0,6.0,1978,0.0,1.0,0.0,0.0,0.0,0.0
4,Empire Strikes Back,538.0,31.0,1980,0.0,0.0,1.0,0.0,0.0,0.0


In [6]:
def estimasi_matriks_ols(X, y, nama_fitur):
    n, p = X.shape
    k = p - 1

    # estimasi parameter OLS: beta_hat = (X^T X)^(-1) X^T Y
    XTX = X.T @ X
    XTX_inv = np.linalg.inv(XTX)
    XTY = X.T @ y
    beta_hat = XTX_inv @ XTY

    y_hat = X @ beta_hat
    e = y - y_hat

    # jumlah kuadrat: JKT = JKR + JKS
    J_mat = np.ones((n, n))
    JKT = float(y.T @ y - (1.0 / n) * (y.T @ J_mat @ y))
    JKR = float(beta_hat.T @ XTY - (1.0 / n) * (y.T @ J_mat @ y))
    JKS = float(y.T @ y - beta_hat.T @ XTY)

    KTR = JKR / k
    MSE = JKS / (n - p)
    R2 = JKR / JKT
    R2_adj = 1.0 - (1.0 - R2) * (n - 1) / (n - p)
    F_hitung = KTR / MSE
    p_val_F = float(1.0 - stats.f.cdf(F_hitung, dfn=k, dfd=n - p))

    # matriks varian-kovarian parameter, statistik t parsial
    cov_beta = MSE * XTX_inv
    se_beta = np.sqrt(np.diag(cov_beta))
    t_hitung = beta_hat / se_beta
    p_val_t = 2.0 * (1.0 - stats.t.cdf(np.abs(t_hitung), df=n - p))

    t_kritis = float(stats.t.ppf(1.0 - ALPHA / 2.0, df=n - p))
    ci_bawah = beta_hat - t_kritis * se_beta
    ci_atas = beta_hat + t_kritis * se_beta

    tabel_koef = pd.DataFrame({
        'Parameter': nama_fitur,
        'Koefisien': beta_hat,
        'Std_Error': se_beta,
        't_hitung': t_hitung,
        'p_value': p_val_t,
        'SK_95%_Bawah': ci_bawah,
        'SK_95%_Atas': ci_atas
    })

    tabel_anava = pd.DataFrame({
        'Sumber_Variansi': ['Regresi', 'Sesatan (Galat)', 'Total'],
        'db': [k, n - p, n - 1],
        'JK': [JKR, JKS, JKT],
        'KT': [KTR, MSE, np.nan],
        'F_hitung': [F_hitung, np.nan, np.nan],
        'p_value': [p_val_F, np.nan, np.nan]
    })

    return {
        'n': n, 'p': p, 'k': k,
        'XTX': XTX, 'XTX_inv': XTX_inv, 'XTY': XTY,
        'beta_hat': beta_hat, 'y_hat': y_hat, 'e': e,
        'JKT': JKT, 'JKR': JKR, 'JKS': JKS, 'KTR': KTR, 'MSE': MSE,
        'R2': R2, 'R2_adj': R2_adj,
        'F_hitung': F_hitung, 'p_val_F': p_val_F,
        'se_beta': se_beta, 't_hitung': t_hitung, 'p_val_t': p_val_t,
        'tabel_koef': tabel_koef, 'tabel_anava': tabel_anava
    }


def jalankan_uji_hipotesis_lengkap(X_full, y_target, res_full, nama_fitur, nama_respon):
    """
    Menjalankan seluruh uji hipotesis formal:
    1. Uji Simultan (Uji F keseluruhan)
    2. Uji Parsial Sub-Vektor Dummy Dekade (Uji F Parsial 5 koefisien dekade)
    3. Uji Parsial Individual (Uji t setiap parameter)
    """
    n, p, k = res_full['n'], res_full['p'], res_full['k']

    # uji F simultan
    F_tabel = float(stats.f.ppf(1.0 - ALPHA, dfn=k, dfd=n - p))
    cetak_uji_hipotesis(
        judul="Uji Signifikansi Regresi Simultan (Uji F)",
        h0="beta_1 = gamma_horror = delta_1970s = ... = delta_2020s = 0",
        h1="Minimal terdapat satu parameter prediktor != 0",
        nama_stat="F_hitung",
        nilai_stat=res_full['F_hitung'],
        kritis_str=f"F_({ALPHA}; {k}, {n - p}) = {F_tabel:.4f}",
        p_val=res_full['p_val_F'],
        ket_tolak=f"Budget, dummy genre horror, dan dummy dekade secara bersama-sama berpengaruh signifikan terhadap {nama_respon}.",
        ket_gagal_tolak=f"Seluruh prediktor secara bersama-sama tidak berpengaruh signifikan terhadap {nama_respon}."
    )

    # uji F parsial kelompok dummy dekade
    X_restricted = X_full[:, :3]  # Intersep, Budget, dan D_horror
    res_rest = estimasi_matriks_ols(X_restricted, y_target, nama_fitur[:3])
    q_dekade = 5
    F_parsial = ((res_rest['JKS'] - res_full['JKS']) / q_dekade) / res_full['MSE']
    p_parsial = float(1.0 - stats.f.cdf(F_parsial, dfn=q_dekade, dfd=n - p))
    F_tabel_part = float(stats.f.ppf(1.0 - ALPHA, dfn=q_dekade, dfd=n - p))

    cetak_uji_hipotesis(
        judul="Uji F Parsial Kelompok Dummy Dekade (5 Kategori Pembanding vs 2010s)",
        h0="delta_1970s = delta_1980s = delta_1990s = delta_2000s = delta_2020s = 0",
        h1="Minimal terdapat satu koefisien dekade != 0",
        nama_stat="F_parsial",
        nilai_stat=F_parsial,
        kritis_str=f"F_({ALPHA}; {q_dekade}, {n - p}) = {F_tabel_part:.4f}",
        p_val=p_parsial,
        ket_tolak=f"Variabel dummy dekade secara bersama-sama memberikan kontribusi signifikan terhadap {nama_respon}.",
        ket_gagal_tolak=f"Variabel dummy dekade secara bersama-sama tidak memberikan kontribusi signifikan terhadap {nama_respon}."
    )

    # uji t parsial individual
    t_tabel = float(stats.t.ppf(1.0 - ALPHA / 2.0, df=n - p))
    for idx, param in enumerate(nama_fitur):
        t_val = res_full['t_hitung'][idx]
        p_val = res_full['p_val_t'][idx]
        koef = res_full['beta_hat'][idx]
        cetak_uji_hipotesis(
            judul=f"Uji Parsial Koefisien '{param}' (Uji t)",
            h0=f"Koefisien '{param}' = 0 (Tidak berpengaruh signifikan secara parsial)",
            h1=f"Koefisien '{param}' != 0 (Berpengaruh signifikan secara parsial)",
            nama_stat="|t_hitung|",
            nilai_stat=abs(t_val),
            kritis_str=f"t_({ALPHA/2}; {n - p}) = {t_tabel:.4f} (t_hitung asli = {t_val:.4f})",
            p_val=p_val,
            ket_tolak=f"Parameter '{param}' ({koef:.4f}) berpengaruh signifikan secara parsial terhadap {nama_respon}.",
            ket_gagal_tolak=f"Parameter '{param}' ({koef:.4f}) tidak berpengaruh signifikan secara parsial pada taraf alpha = {ALPHA}."
        )


def diagnostik_asumsi_klasik(X_matriks, res, nama_prediktor, label="MODEL AWAL"):
    """Menjalankan deteksi pencilan dan uji formal seluruh asumsi klasik."""
    e = res['e']
    s = np.sqrt(res['MSE'])
    n, p = X_matriks.shape
    e_std = e / s

    print("\n" + "=" * 88)
    print(f"DIAGNOSTIK GALAT & UJI ASUMSI KLASIK - {label}")
    print("=" * 88)

    # pencilan: |e_i*| > 3
    idx_out = np.where(np.abs(e_std) > 3.0)[0]
    print(f"[1] Deteksi Pencilan (|e_i*| > 3.0) : {len(idx_out)} observasi -> {df.loc[idx_out, 'title'].tolist()}")

    # normalitas galat: Kolmogorov-Smirnov
    ks_stat, ks_p = stats.kstest(e_std, 'norm')
    cetak_uji_hipotesis(
        judul="Uji Normalitas Galat (Kolmogorov-Smirnov)",
        h0="F_X(x) = F_0(x) (Galat berdistribusi Normal)",
        h1="F_X(x) != F_0(x) (Galat tidak berdistribusi Normal)",
        nama_stat="D_hitung",
        nilai_stat=ks_stat,
        kritis_str=f"Bandingkan p-value dengan alpha = {ALPHA}",
        p_val=ks_p,
        ket_tolak="Asumsi normalitas galat DILANGGAR.",
        ket_gagal_tolak="Asumsi normalitas galat TERPENUHI."
    )

    # homoskedastisitas: korelasi rank Spearman |e_i| vs setiap prediktor
    t_crit_sp = float(stats.t.ppf(1.0 - ALPHA / 2.0, df=n - 2))
    for j in range(1, p):
        r_s, p_s = stats.spearmanr(np.abs(e), X_matriks[:, j])
        t_s = (r_s * np.sqrt(n - 2)) / np.sqrt(max(1.0 - r_s**2, 1e-12))
        cetak_uji_hipotesis(
            judul=f"Uji Homoskedastisitas Rank Spearman (|e_i| vs '{nama_prediktor[j]}')",
            h0=f"rho_s = 0 (Variansi galat homogen terhadap '{nama_prediktor[j]}')",
            h1=f"rho_s != 0 (Terdapat heteroskedastisitas pada '{nama_prediktor[j]}')",
            nama_stat="|t_hitung|",
            nilai_stat=abs(t_s),
            kritis_str=f"t_({ALPHA/2}; {n - 2}) = {t_crit_sp:.4f} (r_s = {r_s:.4f})",
            p_val=p_s,
            ket_tolak=f"Terdeteksi HETEROSKEDASTISITAS pada '{nama_prediktor[j]}'.",
            ket_gagal_tolak=f"Homoskedastisitas TERPENUHI pada '{nama_prediktor[j]}'."
        )

    # independensi galat: Durbin-Watson
    # multikolinieritas: VIF
    dw_stat = float(np.sum(np.diff(e)**2) / np.sum(e**2))
    vif_df = pd.DataFrame({
        'Variabel': nama_prediktor[1:],
        'VIF': [variance_inflation_factor(X_matriks, j) for j in range(1, p)]
    })
    print("-" * 88)
    print(f"[4] Statistik Durbin-Watson : d = {dw_stat:.4f} (Mendekati 2.0 -> Galat independen)")
    print("[5] Variance Inflation Factor (Kriteria VIF < 10):")
    print(vif_df.to_string(index=False))

    return e_std

In [7]:
# %%
hasil_awal = estimasi_matriks_ols(X_mat, Y, feature_names)

print("Tabel ANAVA Model Awal")
display(hasil_awal['tabel_anava'])
print("\nTabel Koefisien Model Awal")
display(hasil_awal['tabel_koef'])
print(f"R^2 = {hasil_awal['R2']:.4f} | Adjusted R^2 = {hasil_awal['R2_adj']:.4f}\n")

jalankan_uji_hipotesis_lengkap(X_mat, Y, hasil_awal, feature_names, nama_respon="pendapatan kotor global (juta USD)")
e_std_awal = diagnostik_asumsi_klasik(X_mat, hasil_awal, feature_names, label="MODEL AWAL")

Tabel ANAVA Model Awal


,Sumber_Variansi,db,JK,KT,F_hitung,p_value
0,Regresi,7,2.942217e+07,4.203167e+06,54.979367,0.0
1,Sesatan (Galat),554,4.235324e+07,7.644989e+04,NaN,NaN
2,Total,561,7.177541e+07,NaN,NaN,NaN



Tabel Koefisien Model Awal


,Parameter,Koefisien,Std_Error,t_hitung,p_value,SK_95%_Bawah,SK_95%_Atas
0,Intersep,214.769016,27.957733,7.681918,7.149836e-14,159.852891,269.685140
1,Budget (juta USD),2.988689,0.178015,16.788938,0.000000e+00,2.639021,3.338357
2,Dummy Horror (1=Horror),-91.262824,46.282672,-1.971857,4.912295e-02,-182.173806,-0.351842
3,Dummy Dekade 1970s,324.719850,140.481689,2.311475,2.117269e-02,48.777952,600.661749
4,Dummy Dekade 1980s,111.776877,75.376849,1.482907,1.386677e-01,-36.282496,259.836251
5,Dummy Dekade 1990s,72.713305,41.008211,1.773140,7.675485e-02,-7.837290,153.263899
6,Dummy Dekade 2000s,-36.475747,28.379628,-1.285279,1.992318e-01,-92.220581,19.269086
7,Dummy Dekade 2020s,-33.623528,53.786452,-0.625130,5.321431e-01,-139.273849,72.026794


R^2 = 0.4099 | Adjusted R^2 = 0.4025

----------------------------------------------------------------------------------------
Uji Hipotesis: Uji Signifikansi Regresi Simultan (Uji F)
  H0: beta_1 = gamma_horror = delta_1970s = ... = delta_2020s = 0
  H1: Minimal terdapat satu parameter prediktor != 0
  Taraf signifikansi: alpha = 0.05
  Statistik uji: F_hitung = 54.9794
  Titik kritis: F_(0.05; 7, 554) = 2.0261
  p-value: 0.0000e+00
  Keputusan: Tolak H0 (karena p-value < 0.05)
  Kesimpulan: Budget, dummy genre horror, dan dummy dekade secara bersama-sama berpengaruh signifikan terhadap pendapatan kotor global (juta USD).
----------------------------------------------------------------------------------------
Uji Hipotesis: Uji F Parsial Kelompok Dummy Dekade (5 Kategori Pembanding vs 2010s)
  H0: delta_1970s = delta_1980s = delta_1990s = delta_2000s = delta_2020s = 0
  H1: Minimal terdapat satu koefisien dekade != 0
  Taraf signifikansi: alpha = 0.05
  Statistik uji: F_parsial = 2.72

In [10]:
# parameter lambda, metode Box-Cox
lambdas = np.linspace(-1.0, 1.0, 9)
K2 = float(np.exp(np.mean(np.log(Y))))
XTX_inv_0 = hasil_awal['XTX_inv']

bc_table = []
for lam in lambdas:
    W = K2 * np.log(Y) if np.isclose(lam, 0.0) else (1.0 / (lam * (K2 ** (lam - 1.0)))) * (Y ** lam - 1.0)
    b_w = XTX_inv_0 @ (X_mat.T @ W)
    bc_table.append({'lambda': round(float(lam), 3), 'JKS_W': float(np.sum((W - X_mat @ b_w)**2))})

print("Tabel Evaluasi Lambda Metode Box-Cox")
display(pd.DataFrame(bc_table))

# model akhir: transformasi log-log
# ln(Y) diregresikan terhadap ln(Budget), D_horror, dan 5 dummy dekade
Y_log = np.log(Y)
names_akhir = [
    'Intersep',
    'ln(Budget)',
    'Dummy Horror (1=Horror)',
    'Dummy Dekade 1970s',
    'Dummy Dekade 1980s',
    'Dummy Dekade 1990s',
    'Dummy Dekade 2000s',
    'Dummy Dekade 2020s'
]
X_mat_akhir = np.column_stack([
    np.ones(n_obs),
    np.log(df['budget_m'].values),
    df['D_horror'].values,
    df[dummy_dekade_cols].values
])

hasil_akhir = estimasi_matriks_ols(X_mat_akhir, Y_log, names_akhir)

print("\nTabel ANAVA Model Akhir")
display(hasil_akhir['tabel_anava'])
print("\nTabel Koefisien Model Akhir")
display(hasil_akhir['tabel_koef'])
print(f"R^2 = {hasil_akhir['R2']:.4f} | Adjusted R^2 = {hasil_akhir['R2_adj']:.4f}\n")

jalankan_uji_hipotesis_lengkap(X_mat_akhir, Y_log, hasil_akhir, names_akhir, nama_respon="ln(pendapatan kotor global)")
e_std_akhir = diagnostik_asumsi_klasik(X_mat_akhir, hasil_akhir, names_akhir, label="MODEL AKHIR (LOG-LOG)")

# persamaan model akhir
b = hasil_akhir['beta_hat']
print("\n" + "=" * 88)
print("Persamaan Model Akhir (Grup dasar: nonhorror, dekade 2010s)")
print("=" * 88)
print(
    f"Bentuk Log-Log: ln(Y_hat) = {b[0]:.4f} + {b[1]:.4f}*ln(Budget) "
    f"{b[2]:+.4f}*D_Horror {b[3]:+.4f}*D_1970s {b[4]:+.4f}*D_1980s "
    f"{b[5]:+.4f}*D_1990s {b[6]:+.4f}*D_2000s {b[7]:+.4f}*D_2020s"
)
print(
    f"Bentuk Eksponensial: Y_hat = exp({b[0]:.4f}) * Budget^{b[1]:.4f} * "
    f"exp({b[2]:.4f}*D_Horror + {b[3]:.4f}*D_1970s + {b[4]:.4f}*D_1980s "
    f"{b[5]:+.4f}*D_1990s {b[6]:+.4f}*D_2000s {b[7]:+.4f}*D_2020s)"
)

Tabel Evaluasi Lambda Metode Box-Cox


,lambda,JKS_W
0,-1.00,2.732196e+09
1,-0.75,5.637350e+08
2,-0.50,1.532243e+08
3,-0.25,5.874147e+07
4,0.00,3.246596e+07
5,0.25,2.512074e+07
6,0.50,2.535106e+07
7,0.75,3.078294e+07
8,1.00,4.235324e+07



Tabel ANAVA Model Akhir


,Sumber_Variansi,db,JK,KT,F_hitung,p_value
0,Regresi,7,236.138237,33.734034,154.503224,0.0
1,Sesatan (Galat),554,120.959642,0.218339,NaN,NaN
2,Total,561,357.097879,NaN,NaN,NaN



Tabel Koefisien Model Akhir


,Parameter,Koefisien,Std_Error,t_hitung,p_value,SK_95%_Bawah,SK_95%_Atas
0,Intersep,3.951520,0.081075,48.738986,0.000000,3.792267,4.110772
1,ln(Budget),0.488560,0.017146,28.493574,0.000000,0.454880,0.522240
2,Dummy Horror (1=Horror),0.137038,0.084289,1.625805,0.104560,-0.028528,0.302604
3,Dummy Dekade 1970s,1.119071,0.237132,4.719199,0.000003,0.653284,1.584858
4,Dummy Dekade 1980s,0.478636,0.125765,3.805801,0.000157,0.231602,0.725670
5,Dummy Dekade 1990s,0.223267,0.067438,3.310709,0.000991,0.090802,0.355732
6,Dummy Dekade 2000s,-0.014983,0.047934,-0.312582,0.754716,-0.109139,0.079172
7,Dummy Dekade 2020s,-0.041427,0.090533,-0.457590,0.647427,-0.219256,0.136403


R^2 = 0.6613 | Adjusted R^2 = 0.6570

----------------------------------------------------------------------------------------
Uji Hipotesis: Uji Signifikansi Regresi Simultan (Uji F)
  H0: beta_1 = gamma_horror = delta_1970s = ... = delta_2020s = 0
  H1: Minimal terdapat satu parameter prediktor != 0
  Taraf signifikansi: alpha = 0.05
  Statistik uji: F_hitung = 154.5032
  Titik kritis: F_(0.05; 7, 554) = 2.0261
  p-value: 0.0000e+00
  Keputusan: Tolak H0 (karena p-value < 0.05)
  Kesimpulan: Budget, dummy genre horror, dan dummy dekade secara bersama-sama berpengaruh signifikan terhadap ln(pendapatan kotor global).
----------------------------------------------------------------------------------------
Uji Hipotesis: Uji F Parsial Kelompok Dummy Dekade (5 Kategori Pembanding vs 2010s)
  H0: delta_1970s = delta_1980s = delta_1990s = delta_2000s = delta_2020s = 0
  H1: Minimal terdapat satu koefisien dekade != 0
  Taraf signifikansi: alpha = 0.05
  Statistik uji: F_parsial = 9.2894
  T

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Konfigurasi global font matplotlib
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Roboto Condensed', 'DejaVu Sans']
plt.rcParams['axes.titleweight'] = 'bold'

folder_output = "tabel"
os.makedirs(folder_output, exist_ok=True)


def simpan_tabel_png(
    df,
    filename,
    judul="",
    folder=folder_output,
    sel_transparan=False,
    warna_teks="#0F172A"
):
    """
    Mengekspor DataFrame menjadi gambar PNG beresolusi 300 dpi dengan background transparan.
    - sel_transparan=False : Latar luar transparan, sel tabel tetap berwarna solid agar mudah dibaca.
    - sel_transparan=True  : Latar luar maupun sel data seluruhnya transparan (alpha=0).
    """
    df_fmt = df.copy()
    for col in df_fmt.select_dtypes(include=[np.number]).columns:
        df_fmt[col] = df_fmt[col].apply(
            lambda x: "-" if pd.isna(x) else (f"{x:.4e}" if (abs(x) < 1e-3 and x != 0) else f"{x:.4f}")
        )

    fig, ax = plt.subplots(figsize=(11, 0.42 * len(df_fmt) + 0.9))

    # Hilangkan background kanvas figure dan axes agar transparan penuh
    fig.patch.set_alpha(0.0)
    ax.patch.set_alpha(0.0)
    ax.axis('off')

    if judul:
        ax.set_title(judul, fontweight='bold', fontsize=11, pad=8, color=warna_teks)

    tbl = ax.table(
        cellText=df_fmt.values,
        colLabels=df_fmt.columns,
        cellLoc='center',
        loc='center'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9.5)
    tbl.auto_set_column_width(col=list(range(len(df_fmt.columns))))
    tbl.scale(1.15, 1.45)

    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            # Header tabel
            cell.set_facecolor('#393939')
            cell.set_text_props(color='white', fontweight='bold')
        else:
            cell.set_text_props(color=warna_teks)
            if sel_transparan:
                cell.set_facecolor('none')
            elif row % 2 == 0:
                cell.set_facecolor('#F8FAFC')
            else:
                cell.set_facecolor('#FFFFFF')
        cell.set_edgecolor('#CBD5E1')

    filepath = os.path.join(folder, filename)
    # Parameter transparent=True memastikan kanal alpha (transparansi) tersimpan di file PNG
    plt.savefig(filepath, dpi=300, bbox_inches='tight', transparent=True)
    plt.close()
    print(f"Tersimpan (transparan): {filepath}")


# Eksekusi penyimpanan ke folder 'tabel'
simpan_tabel_png(hasil_awal['tabel_anava'], 'tabel_anava_awal.png', 'Tabel ANAVA Model Awal')
simpan_tabel_png(hasil_awal['tabel_koef'], 'tabel_koef_awal.png', 'Tabel Koefisien Model Awal')
simpan_tabel_png(hasil_akhir['tabel_anava'], 'tabel_anava_akhir.png', 'Tabel ANAVA Model Akhir (Log-Log)')
simpan_tabel_png(hasil_akhir['tabel_koef'], 'tabel_koef_akhir.png', 'Tabel Koefisien Model Akhir (Log-Log)')

Tersimpan (transparan): tabel\tabel_anava_awal.png
Tersimpan (transparan): tabel\tabel_koef_awal.png
Tersimpan (transparan): tabel\tabel_anava_akhir.png
Tersimpan (transparan): tabel\tabel_koef_akhir.png


In [21]:
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

# Hitung hasil uji Rank Spearman dari galat Model Awal
e_awal = hasil_awal['e']
abs_e = np.abs(e_awal)
n_obs, p_param = X_mat.shape

rows_spearman = []
for j in range(1, p_param):
    r_s, p_s = stats.spearmanr(abs_e, X_mat[:, j])
    t_s = (r_s * np.sqrt(n_obs - 2)) / np.sqrt(1.0 - r_s**2)
    rows_spearman.append({
        'Variabel': feature_names[j],
        'r_s': r_s,
        't_hitung': abs(t_s),
        'p_value': p_s,
        'Keputusan': 'Tolak H0' if p_s < 0.05 else 'Gagal Tolak H0',
        'Keterangan': 'Heteroskedastis' if p_s < 0.05 else 'Homoskedastis'
    })

df_spearman_awal = pd.DataFrame(rows_spearman)

# Simpan ke folder 'tabel' dengan background transparan
simpan_tabel_png(
    df_spearman_awal,
    'tabel_spearman_awal.png',
    judul='Uji Homoskedastisitas Rank Spearman Model Awal (|e_i| vs Prediktor)'
)

Tersimpan (transparan): tabel\tabel_spearman_awal.png


In [23]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Konfigurasi global font matplotlib
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Roboto Condensed', 'DejaVu Sans']
plt.rcParams['axes.titleweight'] = 'bold'


def hitung_tabel_vif(X_matrix, nama_variabel):
    """Menghitung nilai Tolerance dan Variance Inflation Factor (VIF) tiap prediktor."""
    rows = []
    for j in range(1, X_matrix.shape[1]):
        vif_val = float(variance_inflation_factor(X_matrix, j))
        tol_val = 1.0 / vif_val
        rows.append({
            'Variabel': nama_variabel[j],
            'Tolerance': tol_val,
            'VIF': vif_val,
            'Keterangan': 'Bebas Multikolinearitas' if vif_val < 10.0 else 'Terjadi Multikolinearitas'
        })
    return pd.DataFrame(rows)


def simpan_tabel_png(df, filename, judul="", folder="tabel"):
    """Mengekspor DataFrame ke file PNG 300 dpi dengan background transparan."""
    os.makedirs(folder, exist_ok=True)
    df_fmt = df.copy()
    for col in df_fmt.select_dtypes(include=[np.number]).columns:
        df_fmt[col] = df_fmt[col].apply(lambda x: f"{x:.4f}")

    fig, ax = plt.subplots(figsize=(10, 0.42 * len(df_fmt) + 0.9))
    fig.patch.set_alpha(0.0)
    ax.patch.set_alpha(0.0)
    ax.axis('off')

    if judul:
        ax.set_title(judul, fontweight='bold', fontsize=11, pad=8)

    tbl = ax.table(
        cellText=df_fmt.values,
        colLabels=df_fmt.columns,
        cellLoc='center',
        loc='center'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9.5)
    tbl.auto_set_column_width(col=list(range(len(df_fmt.columns))))
    tbl.scale(1.15, 1.45)

    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor('#393939')
            cell.set_text_props(color='white', fontweight='bold')
        elif row % 2 == 0:
            cell.set_facecolor('#F8FAFC')
        else:
            cell.set_facecolor('#FFFFFF')
        cell.set_edgecolor('#CBD5E1')

    filepath = os.path.join(folder, filename)
    plt.savefig(filepath, dpi=300, bbox_inches='tight', transparent=True)
    plt.close()
    print(f"Tersimpan: {filepath}")


# Bentuk tabel VIF Model Awal & Model Akhir
df_vif_awal = hitung_tabel_vif(X_mat, feature_names)
df_vif_akhir = hitung_tabel_vif(X_mat_akhir, names_akhir)

display(df_vif_awal)
display(df_vif_akhir)

# Simpan keduanya ke folder 'tabel/'
simpan_tabel_png(df_vif_awal, 'tabel_vif_awal.png', 'Uji Multikolinearitas Model Awal (Kriteria: Tolerance > 0.1 dan VIF < 10)')
simpan_tabel_png(df_vif_akhir, 'tabel_vif_akhir.png', 'Uji Multikolinearitas Model Akhir Log-Log (Kriteria: Tolerance > 0.1 dan VIF < 10)')

,Variabel,Tolerance,VIF,Keterangan
0,Budget (juta USD),0.743972,1.344137,Bebas Multikolinearitas
1,Dummy Horror (1=Horror),0.797927,1.253248,Bebas Multikolinearitas
2,Dummy Dekade 1970s,0.975393,1.025228,Bebas Multikolinearitas
3,Dummy Dekade 1980s,0.921634,1.085029,Bebas Multikolinearitas
4,Dummy Dekade 1990s,0.860898,1.161578,Bebas Multikolinearitas
5,Dummy Dekade 2000s,0.894463,1.117990,Bebas Multikolinearitas
6,Dummy Dekade 2020s,0.930539,1.074646,Bebas Multikolinearitas


,Variabel,Tolerance,VIF,Keterangan
0,ln(Budget),0.679734,1.471164,Bebas Multikolinearitas
1,Dummy Horror (1=Horror),0.687082,1.455430,Bebas Multikolinearitas
2,Dummy Dekade 1970s,0.977675,1.022835,Bebas Multikolinearitas
3,Dummy Dekade 1980s,0.945520,1.057619,Bebas Multikolinearitas
4,Dummy Dekade 1990s,0.909161,1.099915,Bebas Multikolinearitas
5,Dummy Dekade 2000s,0.895440,1.116769,Bebas Multikolinearitas
6,Dummy Dekade 2020s,0.938044,1.066049,Bebas Multikolinearitas


Tersimpan: tabel\tabel_vif_awal.png
Tersimpan: tabel\tabel_vif_akhir.png
